<a href="https://colab.research.google.com/github/Realbenjizo/python-functions-for-data-engineers/blob/main/Chapter_3_Type_Hints_as_Pipeline_Contracts_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from typing import Dict, Optional, TypedDict

In [ ]:
# Link to the file(Replace with link to your downloaded file)
transactions = "/content/drive/MyDrive/data_files/transactions.csv"

transactions_df = pd.read_csv(transactions, dtype={'user_id': 'Int64'})
print(transactions_df)

   user_id  amount transaction_date     status
0        1  250.50       2025-01-03  completed
1        2    0.00       2025-02-03    pending
2        1  175.75       2025-01-03  completed
3        3   89.99       2025-03-03  completed
4        2  -10.00       2025-02-03     failed
5        4  320.00       2025-04-03  completed
6     <NA>  150.00       2025-01-03  completed
7        1  200.00       2025-05-03  completed


In [ ]:
def costs_calculation(df:pd.DataFrame) -> pd.DataFrame:
    results = df.copy()
    results['cost'] = results['amount'] * 0.25
    return results

costs_calculation(transactions_df)

,user_id,amount,transaction_date,status,cost
0,1,250.50,2025-01-03,completed,62.6250
1,2,0.00,2025-02-03,pending,0.0000
2,1,175.75,2025-01-03,completed,43.9375
3,3,89.99,2025-03-03,completed,22.4975
4,2,-10.00,2025-02-03,failed,-2.5000
5,4,320.00,2025-04-03,completed,80.0000
6,<NA>,150.00,2025-01-03,completed,37.5000
7,1,200.00,2025-05-03,completed,50.0000


In [ ]:
user_lookup: Dict[int, str] = {
     1: "Alice Johnson",
     2: "Bob Smith",
     3: "Carol Williams",
     4: "David Brown"
}

## **Missing Proper Type Hints**

In [ ]:
import pandas as pd

def enrich_user_names(df, lookup=None):
    result_df = df.copy()
    if lookup is None:
        result_df['user_name']= 'Unknown'
        return result_df
    result_df['user_name'] = result_df['user_id'].map(lookup).fillna('Unknown')
    return result_df

print(enrich_user_names(transactions_df, lookup=user_lookup))

   user_id  amount transaction_date     status       user_name
0        1  250.50       2025-01-03  completed   Alice Johnson
1        2    0.00       2025-02-03    pending       Bob Smith
2        1  175.75       2025-01-03  completed   Alice Johnson
3        3   89.99       2025-03-03  completed  Carol Williams
4        2  -10.00       2025-02-03     failed       Bob Smith
5        4  320.00       2025-04-03  completed     David Brown
6     <NA>  150.00       2025-01-03  completed         Unknown
7        1  200.00       2025-05-03  completed   Alice Johnson


## **Proper Type hints and docstring**

In [ ]:
import pandas as pd
from typing import Dict

def enrich_user_names(
        df: pd.DataFrame,
        lookup: Dict[int, str] | None = None) -> pd.DataFrame:
    """Map 'user_id' integers to string names using a lookup dictionary.

    Missing IDs or a missing lookup dictionary default to 'Unknown'.

    Parameters:
        df: DataFrame requiring a 'user_id' column.
        lookup: Dict mapping user IDs (int) to names (str), or None.

    Returns:
        A new DataFrame with an added 'user_name' column.
    """
    # Create a copy of the DataFrame
    result_df = df.copy()

    if lookup is None:
        result_df['user_name'] = 'Unknown'
        return result_df

    result_df['user_name'] = result_df['user_id'].map(lookup).fillna('Unknown')
    return result_df

## **TypedDict for Record-Level Data**

In [ ]:
import pandas as pd
from IPython.display import display # To display results in a notebook
from typing import TypedDict, Optional

class TransactionRecord(TypedDict, total=False):
    """ The expected structure of a single transaction record."""
    user_id: Optional[int]
    amount: float
    transaction_date: pd.Timestamp
    status: str
    user_name: Optional[str]

def enrich_user_name(
    record: TransactionRecord, # Typed dictionary representing one transaction
    user_lookup: Optional[Dict[int, str]] = None
) -> TransactionRecord:
    """Enrich a single transaction record with user name.

    """
    enriched = record.copy() # Make a copy
    if user_lookup is None:
        enriched['user_name'] = 'Unknown'
    else:
        user_id = enriched.get('user_id')
        enriched['user_name'] = user_lookup.get(user_id, 'Unknown')
    return enriched

# Running the function
if __name__ == "__main__":
    # Convert to dict
    sample_record = transactions_df.loc[0].to_dict()
    enriched_record = enrich_user_name(sample_record, user_lookup)
    display(enriched_record)

{'user_id': 1,
 'amount': 250.5,
 'transaction_date': Timestamp('2025-01-03 00:00:00'),
 'status': 'completed',
 'user_name': 'Alice Johnson'}

## **Callable functions**

In [ ]:
def add_tax(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result['amount_with_tax'] = result['amount'] * 1.2
    return result

In [ ]:
from typing import Callable

def apply_transformation(
    df: pd.DataFrame,
    func: Callable[[pd.DataFrame], pd.DataFrame]
) -> pd.DataFrame:
    """Apply a transformation function to a DataFrame."""
    local_df = df.copy()
    return func(local_df)

# Calling the function
if __name__ == "__main__":
    results = apply_transformation(transactions_df, add_tax)
    print(results.head(3))

   user_id  amount transaction_date     status  amount_with_tax
0        1  250.50       2025-01-03  completed            300.6
1        2    0.00       2025-02-03    pending              0.0
2        1  175.75       2025-01-03  completed            210.9


## **TypeVar for Flexible Return Types**

In [ ]:
from typing import TypeVar, Callable

T = TypeVar("T")

def apply_generic(
    df: pd.DataFrame,
    func: Callable[[pd.DataFrame], T]
) -> T:
    """Return whatever type the transformation returns."""
    return func(df.copy())

# calling function
apply_generic(transactions_df, add_tax)

,user_id,amount,transaction_date,status,amount_with_tax
0,1,250.50,2025-01-03,completed,300.600
1,2,0.00,2025-02-03,pending,0.000
2,1,175.75,2025-01-03,completed,210.900
3,3,89.99,2025-03-03,completed,107.988
4,2,-10.00,2025-02-03,failed,-12.000
5,4,320.00,2025-04-03,completed,384.000
6,<NA>,150.00,2025-01-03,completed,180.000
7,1,200.00,2025-05-03,completed,240.000


## Using Pandera for strong Schema

In [ ]:
# Install library
%pip install "pandera[pandas]==0.33.1"

In [ ]:
import pandera.pandas as pa
from pandera.typing import DataFrame, Series
from typing import Optional, Callable
import pandas as pd

class TransactionSchema(pa.DataFrameModel):
    """Schema contract for transformed transaction data."""

    user_id: Series[int] = pa.Field(ge=1, nullable=True)
    amount: Series[float] = pa.Field(nullable=False)
    transaction_date: Series[pd.Timestamp] = pa.Field(nullable=False)
    status: Series[str] = pa.Field(isin=["completed", "failed", "pending"])
    user_name: Optional[str] = pa.Field(nullable=True, default=None)
    amount_with_tax: Series[float] = pa.Field(nullable=False)

@pa.check_output(TransactionSchema)
def add_tax(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result['amount_with_tax'] = result['amount'] * 1.2
    return result

# Calling the function
if __name__ == "__main__":
    results = add_tax(transactions_df)
    print(results.head(3))

   user_id  amount transaction_date     status  amount_with_tax
0        1  250.50       2025-01-03  completed            300.6
1        2    0.00       2025-02-03    pending              0.0
2        1  175.75       2025-01-03  completed            210.9


## **Challenge**

In [ ]:
import pandas as pd
import pandera.pandas as pa
from pandera.typing import DataFrame
from typing import Callable

def calculate_cost(amount: float) -> float:
    """
    Calculate the transaction cost.
    """
    return amount * 0.25

def add_cost_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add a cost column to the transactions DataFrame.
    """
    result = df.copy()
    result["cost"] = result["amount"].apply(calculate_cost)
    return result

class TransactionSchema(pa.DataFrameModel):
    """Pandera schema defining the expected structure and constraints."""

    user_id: int = pa.Field(nullable=True)
    amount: float = pa.Field(nullable=False)
    status: str = pa.Field(isin=["completed", "pending", "failed"])
    cost: float = pa.Field(nullable=False)   # added by transformation


@pa.check_output(TransactionSchema, lazy=True)
def apply_transformation(
    df: pd.DataFrame,
    transformation: Callable[[pd.DataFrame], pd.DataFrame]
) -> DataFrame[TransactionSchema]:
    """
    Apply a DataFrame transformation and validate the output.
    """
    df_copy = df.copy()
    return transformation(df_copy)


if __name__ == "__main__":
    result = apply_transformation(
    transactions_df,
    add_cost_column)
    print("Pipeline executed successfully!")
    print(f"Output shape: {result.shape}")
    print(result)

Pipeline executed successfully!
Output shape: (8, 5)
   user_id  amount transaction_date     status     cost
0        1  250.50       2025-01-03  completed  62.6250
1        2    0.00       2025-02-03    pending   0.0000
2        1  175.75       2025-01-03  completed  43.9375
3        3   89.99       2025-03-03  completed  22.4975
4        2  -10.00       2025-02-03     failed  -2.5000
5        4  320.00       2025-04-03  completed  80.0000
6     <NA>  150.00       2025-01-03  completed  37.5000
7        1  200.00       2025-05-03  completed  50.0000
